In [20]:
class Value:
    def __init__(self,data,_children = (),_op = '' , _label = ''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        self._label = _label
        self.grad = 0

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        out = Value(self.data + other.data , (self,other),'+')
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data , (self,other),'*')
        return out


In [21]:
a = Value(2.0 , _label='a')
b = Value(-3.0 , _label='b')
c = Value(10.0 , _label='c')

In [32]:
e = a*b ; e._label = 'e'

In [23]:
d = e + c
d._label = 'd'

In [24]:
f = Value(-2.0)

In [25]:
L = d*f

In [26]:
L._prev

{Value(data=-2.0), Value(data=4.0)}

In [27]:
L._op

'*'

In [28]:
def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

In [29]:
trace(d)

({Value(data=-3.0),
  Value(data=-6.0),
  Value(data=10.0),
  Value(data=2.0),
  Value(data=4.0)},
 {(Value(data=-3.0), Value(data=-6.0)),
  (Value(data=-6.0), Value(data=4.0)),
  (Value(data=10.0), Value(data=4.0)),
  (Value(data=2.0), Value(data=-6.0))})

In [30]:
from graphviz import Digraph
def draw_dot(root, format='svg', rankdir='LR'):
    """
    format: png | svg | ...
    rankdir: TB (top to bottom graph) | LR (left to right)
    """
    assert rankdir in ['LR', 'TB']
    nodes, edges = trace(root)
    dot = Digraph(format=format, graph_attr={'rankdir': rankdir}) #, node_attr={'rankdir': 'TB'})

    for n in nodes:
        dot.node(name=str(id(n)), label = "{ %s | data %.4f| grad %.4f }" % (n._label ,n.data,n.grad), shape='record')
        if n._op:
            dot.node(name=str(id(n)) + n._op, label=n._op)
            dot.edge(str(id(n)) + n._op, str(id(n)))

    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

In [31]:
draw_dot(d)

ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH

In [33]:
def lol():
    h = 0.0001

    a = Value(2.0 , _label='a')
    b = Value(-3.0 , _label='b')
    c = Value(10.0 , _label='c')
    f = Value(-2.0 ,_label= 'f')
    e = a*b ;e._label = 'e'
    d = e + c;d._label = 'd'
    L = d*f; L._label = 'L'
    L1 = L.data

    a = Value(2.0 + h , _label='a')
    b = Value(-3.0 , _label='b')
    c = Value(10.0 , _label='c')
    f = Value(-2.0 ,_label= 'f')
    e = a*b ;e._label = 'e'
    d = e + c;d._label = 'd'
    L = d*f; L._label = 'L'
    L2 = L.data

    print((L2-L1)/h)



In [34]:
lol()

6.000000000021544
